# REST와 페이지네이션

컴퓨터에서 실행한 HTTP 서버로부터 두 페이지를 순서대로 받습니다.

## 실행 환경 준비

다음 두 코드 셀은 HTTP 요청과 응답을 관찰하기 위한 로컬 보조 서버를 만듭니다. 서버 구현 자체는 학습 목표가 아닙니다. `json`은 Python 값을 JSON 본문으로 바꾸고, `threading`은 서버가 요청을 기다리는 동안 다음 셀을 실행할 수 있도록 별도 실행 흐름을 제공합니다.

In [1]:
# 실제 로컬 HTTP 서버와 httpx 클라이언트를 준비합니다.
import json
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import httpx

pages = {1:{"items":[{"id":"A01"},{"id":"A02"}],"next_page":2},
         2:{"items":[{"id":"A03"}],"next_page":None}}

## 기본 실행

`class Handler(BaseHTTPRequestHandler)`의 괄호 안 이름은 기존 HTTP 처리 기능을 물려받는 부모 클래스입니다. 이 상속 표기는 아직 일반 Python 경로에서 배우지 않았으므로 그대로 실행할 보조 코드로 읽습니다. `do_GET`은 GET 요청을 처리하고, `send_response`와 `send_header`는 상태와 헤더를, `wfile.write`는 JSON 본문을 보냅니다. `ThreadingHTTPServer(("127.0.0.1", 0), Handler)`의 포트 0은 운영체제에 사용 가능한 포트를 선택해 달라는 뜻입니다. `Thread(...).start()`는 서버 대기를 별도 실행 흐름에서 시작합니다.

In [2]:
# 요청의 page 값을 읽어 결정적인 JSON 페이지를 반환합니다.
class Handler(BaseHTTPRequestHandler):
    def do_GET(self) -> None:
        page = int(self.path.split("page=")[-1])
        body = json.dumps(pages[page]).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)
    def log_message(self, format: str, *args: object) -> None:
        pass
server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
base_url = f"http://127.0.0.1:{server.server_port}"
print("로컬 서버 준비 완료")

로컬 서버 준비 완료


## 중간 결과 관찰

첫 GET 요청의 입력은 `/articles` 주소와 `page=1` 쿼리 매개변수입니다. `raise_for_status()`로 오류 상태를 먼저 검사하고 `response.json()`으로 JSON 본문을 Python 사전으로 바꿉니다. 출력의 `items`는 현재 페이지 자료이고 `next_page=2`는 다음 요청에 사용할 입력입니다.

In [3]:
# 첫 페이지의 상태를 확인한 뒤 JSON과 다음 페이지 값을 읽습니다.
response = httpx.get(f"{base_url}/articles", params={"page":1}, timeout=2.0)
response.raise_for_status()
first = response.json()
print("상태 코드:", response.status_code)
print("요청 주소:", response.request.url)
print(first)
assert first["next_page"] == 2

상태 코드: 200
요청 주소: http://127.0.0.1:54376/articles?page=1
{'items': [{'id': 'A01'}, {'id': 'A02'}], 'next_page': 2}


## 값 하나를 바꿔 비교

페이지 입력만 1에서 2로 바꿉니다. JSON의 `null`은 Python에서 `None`으로 변환되므로 `next_page is None`이면 더 요청할 페이지가 없다는 뜻입니다.

In [4]:
# page 값만 바꾸어 두 번째 응답이 종료를 표시하는지 확인합니다.
second = httpx.get(f"{base_url}/articles", params={"page":2}, timeout=2.0).json()
print(second)
assert second["next_page"] is None

{'items': [{'id': 'A03'}], 'next_page': None}


## 실패 재현

첫 응답만 반환하면 두 번째 페이지의 A03이 누락됩니다.

In [5]:
# 첫 응답만 반환하는 함수는 뒤쪽 기사 A03을 누락합니다.
def fetch_one_page(base_url: str) -> list[dict]:
    response = httpx.get(f"{base_url}/articles", params={"page":1}, timeout=2.0)
    response.raise_for_status()
    return response.json()["items"]
wrong = fetch_one_page(base_url)
print("한 페이지 ID:", [item["id"] for item in wrong])
assert [item["id"] for item in wrong] == ["A01", "A02"]

한 페이지 ID: ['A01', 'A02']


## 수정

이 노트북의 기본 함수는 `base_url`과 `timeout`만 입력받습니다. `httpx.Client(timeout=timeout)`은 여러 요청에 재사용할 클라이언트 객체를 반환합니다. `client.get(주소, params=값)`은 GET 요청을 보내 `Response` 객체를 반환합니다.

`items.extend(현재_목록)`은 현재 페이지의 각 기사를 기존 목록 뒤에 추가합니다. `payload.get(...)`은 다음 페이지 번호를 반환하며 키가 없으면 `None`을 반환합니다. `None`이 될 때까지 요청한 뒤 기사 목록을 반환합니다. `finally`의 `client.close()`는 오류가 발생해도 연결 자원을 정리합니다. 공용 `data_tools.py`에는 범주 필터, 외부 클라이언트 입력, 반복 페이지 검사도 있습니다.

In [6]:
# 기본 요청, 누적, 다음 페이지 갱신, 반환 순서로 구현합니다.
def fetch_articles(base_url: str, *, timeout: float = 2.0) -> list[dict]:
    client = httpx.Client(timeout=timeout)
    items: list[dict] = []
    page: int | None = 1
    try:
        while page is not None:
            response = client.get(f"{base_url}/articles", params={"page": page})
            response.raise_for_status()
            payload = response.json()
            items.extend(payload["items"])
            page = payload.get("next_page")
    finally:
        client.close()
    return items
all_items = fetch_articles(base_url)
print("전체 ID:", [item["id"] for item in all_items])
assert [item["id"] for item in all_items] == ["A01", "A02", "A03"]
server.shutdown()
thread.join()

전체 ID: ['A01', 'A02', 'A03']
